In [1]:

import pandas as pd
from pathlib import Path

INPUT_DIR = Path("data/validated")
OUTPUT_DIR = Path("data/quality_checked")
REPORT_DIR = Path("output/reports")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "cuti": INPUT_DIR / "cuti_validated.xlsx",
    "tanggal": INPUT_DIR / "tanggal_validated.xlsx"
}

BUSINESS_KEY = ["employee_id", "attendance_date"]


def validate_business_rules(df):
    df = df.copy()

    # Preserve original records and their order.
    df["source_row_number"] = range(2, len(df) + 2)

    data_columns = [
        c for c in df.columns
        if c != "source_row_number"
    ]

    # Duplicate checks
    df["is_full_duplicate"] = df.duplicated(
        subset=data_columns, keep=False
    )

    df["is_business_key_duplicate"] = (
        df.duplicated(BUSINESS_KEY, keep=False)
    )

    # Required fields
    df["missing_employee_id"] = (
        df["employee_id"].isna()
        | df["employee_id"].astype("string").str.strip().eq("").fillna(False)
    )

    df["missing_attendance_date"] = (
        df["attendance_date"].isna()
    )

    # Attendance status is not necessarily required.
    if "attendance_type" in df.columns:
        df["missing_attendance_type"] = (
            df["attendance_type"].isna()
        )
    else:
        df["missing_attendance_type"] = False

    # Do not interpret a missing clock-in as absence.
    for col in ["clock_in_time", "clock_out_time"]:
        if col in df.columns:
            df[f"missing_{col}"] = df[col].isna()
        else:
            df[f"missing_{col}"] = False

    # Classification based on critical checks.
    critical = (
        df["missing_employee_id"]
        | df["missing_attendance_date"]
    )

    duplicate = (
        df["is_full_duplicate"]
        | df["is_business_key_duplicate"]
    )

    df["quality_status"] = "VALID"
    df.loc[duplicate, "quality_status"] = "REVIEW"
    df.loc[critical, "quality_status"] = "INVALID"

    # Missing clock-in/out and attendance type are
    # reported separately, not automatically invalid.

    return df


summary = []
all_duplicates = []

for source, path in FILES.items():
    df = pd.read_excel(
        path,
        dtype={"employee_id": "string"}
    )

    result = validate_business_rules(df)

    duplicate_rows = result[
        result["is_full_duplicate"]
        | result["is_business_key_duplicate"]
    ].copy()

    duplicate_rows.insert(0, "source", source)
    all_duplicates.append(duplicate_rows)

    summary.append({
        "source": source,
        "total_rows": len(result),
        "full_duplicate_rows": int(
            result["is_full_duplicate"].sum()
        ),
        "business_key_duplicate_rows": int(
            result["is_business_key_duplicate"].sum()
        ),
        "missing_employee_id": int(
            result["missing_employee_id"].sum()
        ),
        "missing_attendance_date": int(
            result["missing_attendance_date"].sum()
        ),
        "valid_rows": int(
            (result["quality_status"] == "VALID").sum()
        ),
        "review_rows": int(
            (result["quality_status"] == "REVIEW").sum()
        ),
        "invalid_rows": int(
            (result["quality_status"] == "INVALID").sum()
        )
    })

    result.to_excel(
        OUTPUT_DIR / f"{source}_quality_checked.xlsx",
        index=False
    )

summary_df = pd.DataFrame(summary)

summary_df.to_excel(
    REPORT_DIR / "data_quality_summary.xlsx",
    index=False
)

duplicates_df = pd.concat(
    all_duplicates, ignore_index=True
)

duplicates_df.to_excel(
    REPORT_DIR / "duplicate_review.xlsx",
    index=False
)

print(summary_df.to_string(index=False))
print("\nBusiness validation complete.")
print("No records deleted. No merge performed.")


 source  total_rows  full_duplicate_rows  business_key_duplicate_rows  missing_employee_id  missing_attendance_date  valid_rows  review_rows  invalid_rows
   cuti        7415                    0                            0                    0                        0        7415            0             0
tanggal       27774                    0                          126                    0                        0       27648          126             0

Business validation complete.
No records deleted. No merge performed.


In [2]:

import pandas as pd
from pathlib import Path

INPUT_FILE = Path(
    "data/quality_checked/tanggal_quality_checked.xlsx"
)

OUTPUT_FILE = Path(
    "output/reports/duplicate_investigation.xlsx"
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(
    INPUT_FILE,
    dtype={"employee_id": "string"}
)

KEY = ["employee_id", "attendance_date"]

# Identify duplicate groups
duplicates = df[
    df.duplicated(KEY, keep=False)
].copy()

duplicates = duplicates.sort_values(KEY)

# Exclude technical validation columns
technical_columns = [
    "source_row_number",
    "is_full_duplicate",
    "is_business_key_duplicate",
    "missing_employee_id",
    "missing_attendance_date",
    "missing_attendance_type",
    "missing_clock_in_time",
    "missing_clock_out_time",
    "quality_status"
]

compare_columns = [
    col for col in duplicates.columns
    if col not in KEY + technical_columns
]

group_summary = (
    duplicates.groupby(KEY, dropna=False)
    .size()
    .reset_index(name="record_count")
)

# Find which fields differ within each group
differences = []

for keys, group in duplicates.groupby(
    KEY, dropna=False, sort=False
):
    employee_id, attendance_date = keys

    different_columns = []

    for col in compare_columns:
        if group[col].nunique(dropna=False) > 1:
            different_columns.append(col)

    differences.append({
        "employee_id": employee_id,
        "attendance_date": attendance_date,
        "record_count": len(group),
        "different_column_count": len(different_columns),
        "different_columns": ", ".join(different_columns)
    })

difference_report = pd.DataFrame(differences)

# Count how often each column differs
column_difference_summary = []

for col in compare_columns:
    count = sum(
        col in group_cols
        for group_cols in [
            row.split(", ") if row else []
            for row in difference_report["different_columns"]
        ]
    )

    if count > 0:
        column_difference_summary.append({
            "column": col,
            "different_groups": count
        })

column_difference_summary = pd.DataFrame(
    column_difference_summary,
    columns=["column", "different_groups"]
).sort_values("different_groups", ascending=False)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    duplicates.to_excel(
        writer, sheet_name="Duplicate Records", index=False
    )

    group_summary.to_excel(
        writer, sheet_name="Group Summary", index=False
    )

    difference_report.to_excel(
        writer, sheet_name="Field Differences", index=False
    )

    column_difference_summary.to_excel(
        writer, sheet_name="Difference Frequency", index=False
    )

print("Duplicate records:", len(duplicates))
print("Duplicate groups:", len(group_summary))
print("\nDifference frequency:")
print(column_difference_summary.to_string(index=False))
print("\nReport saved:", OUTPUT_FILE)


Duplicate records: 126
Duplicate groups: 63

Difference frequency:
           column  different_groups
       updated_by                63
      last_update                63
           status                63
attendance_source                62
    attendance_in                51
 total_work_hours                38
    clock_in_time                37
   clock_out_time                37
clock_out_time_be                37
 clock_in_time_be                37
        shift_out                 7
         shift_in                 7
            shift                 7
  attendance_type                 7

Report saved: output\reports\duplicate_investigation.xlsx


In [3]:
BUSINESS_KEY = [
    "employee_id",
    "attendance_date",
    "shift_in",
    "shift_out"
]


import pandas as pd
from pathlib import Path

INPUT_DIR = Path("data/quality_checked")
OUTPUT_DIR = Path("data/deduplicated")
REPORT_DIR = Path("output/reports")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "tanggal": INPUT_DIR / "tanggal_quality_checked.xlsx"
}

KEY = [
    "employee_id",
    "attendance_date",
    "shift_in",
    "shift_out"
]

# HDR and HOVT are expected in attendance_in.
STATUS_COLUMN = "attendance_in"

for source, file_path in FILES.items():
    df = pd.read_excel(
        file_path,
        dtype={"employee_id": "string"}
    )

    required = KEY + [STATUS_COLUMN]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df = df.copy()

    # Preserve input order
    df["_original_order"] = range(len(df))

    # Normalize HDR/HOVT
    df["_status"] = (
        df[STATUS_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Normalize keys without changing the original columns
    df["_employee_key"] = (
        df["employee_id"].astype("string").str.strip()
    )

    df["_date_key"] = pd.to_datetime(
        df["attendance_date"], errors="coerce"
    ).dt.normalize()

    for col in ["shift_in", "shift_out"]:
        parsed = pd.to_datetime(
            df[col].astype("string").str.strip(),
            format="%H:%M:%S",
            errors="coerce"
        )

        fallback = pd.to_datetime(
            df[col].astype("string").str.strip(),
            format="%H:%M",
            errors="coerce"
        )

        parsed = parsed.fillna(fallback)
        df[f"_{col}_key"] = parsed.dt.strftime("%H:%M:%S")

    group_key = [
        "_employee_key",
        "_date_key",
        "_shift_in_key",
        "_shift_out_key"
    ]

    # Incomplete keys must be reviewed, not deduplicated.
    valid_key = df[group_key].notna().all(axis=1)

    df["dedup_action"] = "KEEP"
    df["dedup_rule"] = "NO_MATCH"

    df.loc[~valid_key, "dedup_rule"] = "REVIEW_KEY"

    # Rule R01: HDR wins over HOVT for matching key.
    has_hdr = (
        df["_status"].eq("HDR")
        .groupby([df[k] for k in group_key], dropna=False)
        .transform("any")
    )

    drop_hovt = (
        valid_key
        & df["_status"].eq("HOVT")
        & has_hdr
    )

    df.loc[drop_hovt, "dedup_action"] = "DROP"
    df.loc[drop_hovt, "dedup_rule"] = "R01_HOVT_WITH_HDR"

    # Rule R02: Keep the first HDR per exact key.
    hdr_rows = df.loc[
        valid_key & df["_status"].eq("HDR")
    ]

    duplicated_hdr = hdr_rows.duplicated(
        subset=group_key,
        keep="first"
    )

    hdr_drop_indices = hdr_rows.index[duplicated_hdr]

    df.loc[hdr_drop_indices, "dedup_action"] = "DROP"
    df.loc[hdr_drop_indices, "dedup_rule"] = "R02_DUPLICATE_HDR"

    # Separate final data and deleted records.
    kept = df[df["dedup_action"] == "KEEP"].copy()
    dropped = df[df["dedup_action"] == "DROP"].copy()

    helper_cols = [
        "_original_order", "_status",
        "_employee_key", "_date_key",
        "_shift_in_key", "_shift_out_key"
    ]

    kept = kept.drop(columns=helper_cols)
    dropped = dropped.drop(columns=helper_cols)

    # Save results
    kept.to_excel(
        OUTPUT_DIR / f"{source}_deduplicated.xlsx",
        index=False
    )

    dropped.to_excel(
        REPORT_DIR / f"{source}_deleted_records.xlsx",
        index=False
    )

    df["dedup_rule"].value_counts().rename_axis(
        "rule"
    ).reset_index(name="count").to_excel(
        REPORT_DIR / f"{source}_dedup_summary.xlsx",
        index=False
    )

    assert len(df) == len(kept) + len(dropped)

    print(f"\nDataset: {source}")
    print(f"Original rows: {len(df):,}")
    print(f"Kept rows: {len(kept):,}")
    print(f"Deleted rows: {len(dropped):,}")
    print("\nRules applied:")
    print(df["dedup_rule"].value_counts())

print("\nDeduplication completed.")
print("No merge performed.")



Dataset: tanggal
Original rows: 27,774
Kept rows: 27,774
Deleted rows: 0

Rules applied:
dedup_rule
REVIEW_KEY    20732
NO_MATCH       7042
Name: count, dtype: int64

Deduplication completed.
No merge performed.
